### Game Dashboard Setup

Creates an AI/BI Dashboard for Level 2 of Casper's Kitchen Rescue:
- **Level 2**: Delivery timing dashboard (The Slow Kitchen)

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

##### Create dashboard via Lakeview API

We create the dashboard programmatically with pre-built queries that
surface the delivery timing data. Players need to interact with the
charts and filters to find the anomaly.

In [ ]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

# Get warehouse
warehouses = list(w.warehouses.list())
warehouse_id = None
for wh in warehouses:
    if wh.name and CATALOG in wh.name:
        warehouse_id = wh.id
        break
if not warehouse_id:
    for wh in warehouses:
        if wh.enable_serverless_compute or 'serverless' in (wh.name or '').lower():
            warehouse_id = wh.id
            break
if not warehouse_id and warehouses:
    warehouse_id = warehouses[0].id

print(f"Using warehouse_id={warehouse_id}")

In [ ]:
# Define the Lakeview dashboard serialized definition
# This creates a multi-panel dashboard for the "Slow Kitchen" investigation

dashboard_definition = {
    "pages": [
        {
            "name": "investigation",
            "displayName": "Kitchen Investigation",
            "layout": [
                {
                    "widget": {
                        "name": "narrative",
                        "textbox_spec": {
                            "value": "## \ud83d\udd0d The Slow Kitchen Investigation\n\nCustomers are complaining about cold food. One kitchen is significantly slower than others. **Use the charts below to identify:**\n1. Which location has abnormally long kitchen prep times?\n2. When did the slowdown start?\n\n*Tip: Compare the kitchen prep time trends across locations. Look for a divergence.*"
                        }
                    },
                    "position": {"x": 0, "y": 0, "width": 6, "height": 2}
                },
                {
                    "widget": {
                        "name": "avg_prep_by_location",
                        "queries": [
                            {
                                "name": "avg_prep",
                                "query": {
                                    "datasetName": "delivery_times",
                                    "disaggregated": False,
                                    "fields": [
                                        {"name": "location_name", "expression": "`location_name`"},
                                        {"name": "avg_kitchen_prep", "expression": "AVG(`kitchen_prep_minutes`)", "type": "number"}
                                    ]
                                }
                            }
                        ],
                        "spec": {
                            "version": 3,
                            "widgetType": "bar",
                            "encodings": {
                                "x": {"fieldName": "location_name", "displayName": "Location"},
                                "y": {"fieldName": "avg_kitchen_prep", "displayName": "Avg Kitchen Prep (min)"}
                            }
                        }
                    },
                    "position": {"x": 0, "y": 2, "width": 3, "height": 3}
                },
                {
                    "widget": {
                        "name": "avg_delivery_by_location",
                        "queries": [
                            {
                                "name": "avg_delivery",
                                "query": {
                                    "datasetName": "delivery_times",
                                    "disaggregated": False,
                                    "fields": [
                                        {"name": "location_name", "expression": "`location_name`"},
                                        {"name": "avg_total_delivery", "expression": "AVG(`total_delivery_minutes`)", "type": "number"}
                                    ]
                                }
                            }
                        ],
                        "spec": {
                            "version": 3,
                            "widgetType": "bar",
                            "encodings": {
                                "x": {"fieldName": "location_name", "displayName": "Location"},
                                "y": {"fieldName": "avg_total_delivery", "displayName": "Avg Total Delivery (min)"}
                            }
                        }
                    },
                    "position": {"x": 3, "y": 2, "width": 3, "height": 3}
                },
                {
                    "widget": {
                        "name": "prep_trend_by_day",
                        "queries": [
                            {
                                "name": "daily_prep_trend",
                                "query": {
                                    "datasetName": "delivery_times",
                                    "disaggregated": False,
                                    "fields": [
                                        {"name": "order_day", "expression": "`order_day`"},
                                        {"name": "location_name", "expression": "`location_name`"},
                                        {"name": "avg_prep", "expression": "AVG(`kitchen_prep_minutes`)", "type": "number"}
                                    ]
                                }
                            }
                        ],
                        "spec": {
                            "version": 3,
                            "widgetType": "line",
                            "encodings": {
                                "x": {"fieldName": "order_day", "displayName": "Date"},
                                "y": {"fieldName": "avg_prep", "displayName": "Avg Kitchen Prep (min)"},
                                "color": {"fieldName": "location_name", "displayName": "Location"}
                            }
                        }
                    },
                    "position": {"x": 0, "y": 5, "width": 6, "height": 3}
                },
                {
                    "widget": {
                        "name": "delivery_trend_by_day",
                        "queries": [
                            {
                                "name": "daily_delivery_trend",
                                "query": {
                                    "datasetName": "delivery_times",
                                    "disaggregated": False,
                                    "fields": [
                                        {"name": "order_day", "expression": "`order_day`"},
                                        {"name": "location_name", "expression": "`location_name`"},
                                        {"name": "avg_delivery", "expression": "AVG(`total_delivery_minutes`)", "type": "number"}
                                    ]
                                }
                            }
                        ],
                        "spec": {
                            "version": 3,
                            "widgetType": "line",
                            "encodings": {
                                "x": {"fieldName": "order_day", "displayName": "Date"},
                                "y": {"fieldName": "avg_delivery", "displayName": "Avg Total Delivery (min)"},
                                "color": {"fieldName": "location_name", "displayName": "Location"}
                            }
                        }
                    },
                    "position": {"x": 0, "y": 8, "width": 6, "height": 3}
                }
            ]
        }
    ],
    "datasets": [
        {
            "name": "delivery_times",
            "displayName": "Delivery Times",
            "query": f"SELECT * FROM {CATALOG}.game.investigation_delivery_times"
        }
    ]
}

print("Dashboard definition ready")
print(f"  Datasets: {[d['name'] for d in dashboard_definition['datasets']]}")
print(f"  Widgets: {[w['widget']['name'] for w in dashboard_definition['pages'][0]['layout']]}")

In [ ]:
host = w.config.host.rstrip("/")
dash_title = f"Casper's Kitchen Investigation - The Slow Kitchen ({CATALOG})"
final_url = None
dashboard_id = None

# 1. Check for existing dashboard
print("Checking for existing dashboard...")
try:
    existing = list(w.lakeview.list())
    for d in existing:
        if d.display_name == dash_title:
            dashboard_id = d.dashboard_id
            final_url = f"{host}/sql/dashboardsv3/{dashboard_id}"
            print(f"♻️ Reusing existing dashboard: {final_url}")
            break
except Exception as e:
    print(f"Could not list dashboards: {e}")

# 2. Create via SDK
if not final_url:
    print("Creating dashboard via SDK...")
    try:
        dashboard = w.lakeview.create(
            display_name=dash_title,
            warehouse_id=warehouse_id,
            serialized_dashboard=json.dumps(dashboard_definition)
        )
        dashboard_id = dashboard.dashboard_id
        final_url = f"{host}/sql/dashboardsv3/{dashboard_id}"
        print(f"\u2705 Created dashboard: {final_url}")
    except Exception as e:
        print(f"\u26a0\ufe0f SDK create failed: {e}")

# 3. Fallback: REST API
if not final_url:
    print("Trying REST API fallback...")
    try:
        token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    except Exception:
        token = w.config.token
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    try:
        import requests
        resp = requests.post(
            f"{host}/api/2.0/lakeview/dashboards",
            headers=headers,
            json={
                "display_name": dash_title,
                "warehouse_id": warehouse_id,
                "serialized_dashboard": json.dumps(dashboard_definition),
            },
        )
        print(f"  Response status: {resp.status_code}")
        print(f"  Response body: {resp.text[:500]}")
        resp.raise_for_status()
        dashboard_id = resp.json().get("dashboard_id")
        if dashboard_id:
            final_url = f"{host}/sql/dashboardsv3/{dashboard_id}"
            print(f"\u2705 Created dashboard via REST: {final_url}")
    except Exception as e:
        print(f"\u26a0\ufe0f REST API also failed: {e}")

# 4. Publish if we have a dashboard (draft URL already works for logged-in users)
if dashboard_id:
    try:
        w.lakeview.publish(
            dashboard_id=dashboard_id,
            warehouse_id=warehouse_id,
            embed_credentials=True,
        )
        print(f"\u2705 Published dashboard (URL: {final_url})")
    except Exception as pub_err:
        print(f"\u26a0\ufe0f Could not publish (draft URL will be used): {pub_err}")

if not final_url:
    print("\n\u274c Could not create dashboard.")
    print(f"Create one manually, then run:")
    print(f"  MERGE INTO {CATALOG}.game.config AS t")
    print(f"  USING (SELECT 'dashboard_url' AS config_key, '<URL>' AS config_value) AS s")
    print(f"  ON t.config_key = s.config_key")
    print(f"  WHEN MATCHED THEN UPDATE SET config_value = s.config_value")
    print(f"  WHEN NOT MATCHED THEN INSERT VALUES (s.config_key, s.config_value);")


In [ ]:
# Store dashboard URL in game config
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.game.config (
  config_key STRING,
  config_value STRING
)
""")

if final_url:
    spark.sql(f"""
    MERGE INTO {CATALOG}.game.config AS target
    USING (SELECT 'dashboard_url' AS config_key, '{final_url}' AS config_value) AS source
    ON target.config_key = source.config_key
    WHEN MATCHED THEN UPDATE SET config_value = source.config_value
    WHEN NOT MATCHED THEN INSERT (config_key, config_value) VALUES (source.config_key, source.config_value)
    """)
    print(f"\u2705 Stored dashboard URL in game config")
else:
    print("\u26a0\ufe0f No dashboard URL to store. Set it manually in game.config.")

In [ ]:
print("\u2705 Dashboard setup complete")